In [0]:
from pyspark.sql.functions import (
    col,
    date_format,
    to_date
)

SILVER_TABLE = "workspace.default.silver_hvfhv_trips"
LOCATION_TABLE = "workspace.default.dim_location"
PROVIDER_TABLE = "workspace.default.dim_provider"
DATE_TABLE = "workspace.default.dim_date"

df_silver = spark.table(SILVER_TABLE)
df_location = spark.table(LOCATION_TABLE)
df_provider = spark.table(PROVIDER_TABLE)
df_date = spark.table(DATE_TABLE)

print("Silver rows:", df_silver.count())
print("Location rows:", df_location.count())
print("Provider rows:", df_provider.count())
print("Date rows:", df_date.count())

In [0]:
from pyspark.sql.functions import xxhash64

df_fact_base = (
    df_silver
    .withColumn(
        "trip_key",
        xxhash64(
            "hvfhs_license_num",
            "request_datetime",
            "pickup_datetime",
            "dropoff_datetime",
            "PULocationID",
            "DOLocationID",
            "trip_miles",
            "trip_time",
            "base_passenger_fare",
            "driver_pay"
        )
    )
)

In [0]:
df_fact_base = (
    df_fact_base
    .withColumn(
        "date_key",
        date_format(
            to_date("pickup_datetime"),
            "yyyyMMdd"
        ).cast("integer")
    )
)

In [0]:
df_fact_base = (
    df_fact_base
    .withColumn(
        "provider_key",
        col("hvfhs_license_num")
    )
)

In [0]:
df_fact_base = (
    df_fact_base
    .withColumn(
        "pickup_location_key",
        col("PULocationID").cast("integer")
    )
    .withColumn(
        "dropoff_location_key",
        col("DOLocationID").cast("integer")
    )
)

In [0]:
fact_columns = [
    "trip_key",

    # Dimension keys
    "date_key",
    "provider_key",
    "pickup_location_key",
    "dropoff_location_key",

    # Timestamps
    "request_datetime",
    "on_scene_datetime",
    "pickup_datetime",
    "dropoff_datetime",

    # Trip measures
    "trip_miles",
    "trip_time",

    # Operational measures
    "customer_wait_seconds",
    "driver_response_seconds",
    "calculated_trip_time_seconds",

    # Financial measures
    "base_passenger_fare",
    "tolls",
    "bcf",
    "sales_tax",
    "congestion_surcharge",
    "airport_fee",
    "tips",
    "driver_pay",
    "cbd_congestion_fee",

    # Flags
    "shared_request",
    "shared_match",
    "access_a_ride",
    "wav_request",
    "wav_match",

    # Quality
    "trip_time_consistent",
    "timestamp_quality_flag",
    "financial_quality_flag",
    "overall_quality_status",

    # Lineage
    "_source_month",
    "_source_file",
    "_ingested_at"
]

df_fact_trip = df_fact_base.select(fact_columns)

In [0]:
print("Fact rows:", df_fact_trip.count())
print("Fact columns:", len(df_fact_trip.columns))

In [0]:
display(
    df_fact_trip
    .groupBy("trip_key")
    .count()
    .filter(col("count") > 1)
    .limit(20)
)

In [0]:
print(
    "Null date keys:",
    df_fact_trip.filter(col("date_key").isNull()).count()
)

print(
    "Null provider keys:",
    df_fact_trip.filter(col("provider_key").isNull()).count()
)

print(
    "Null pickup keys:",
    df_fact_trip.filter(col("pickup_location_key").isNull()).count()
)

print(
    "Null dropoff keys:",
    df_fact_trip.filter(col("dropoff_location_key").isNull()).count()
)

In [0]:
display(
    df_fact_trip
    .select(
        to_date("pickup_datetime").alias("pickup_date")
    )
    .agg(
        {"pickup_date": "min"}
    )
)

display(
    df_fact_trip
    .select(
        to_date("pickup_datetime").alias("pickup_date")
    )
    .agg(
        {"pickup_date": "max"}
    )
)

In [0]:
FACT_TABLE = "workspace.default.fact_trip"

(
    df_fact_trip
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("_source_month")
    .saveAsTable(FACT_TABLE)
)

In [0]:
df_fact = spark.table(FACT_TABLE)

print("Fact rows:", df_fact.count())
print("Fact columns:", len(df_fact.columns))

In [0]:
display(
    df_fact
    .select(
        "trip_key",
        "date_key",
        "provider_key",
        "pickup_location_key",
        "dropoff_location_key",
        "pickup_datetime",
        "dropoff_datetime",
        "trip_miles",
        "trip_time"
    )
    .limit(10)
)

In [0]:
from pyspark.sql.functions import col

display(
    df_fact.alias("fact")
    .join(
        df_provider.alias("provider"),
        col("fact.provider_key") == col("provider.provider_key"),
        "left"
    )
    .filter(col("provider.provider_key").isNull())
    .select(col("fact.provider_key").alias("provider_key"))
    .distinct()
)

In [0]:
display(
    df_fact
    .join(
        df_location,
        df_fact.pickup_location_key == df_location.location_key,
        "left"
    )
    .filter(df_location.location_key.isNull())
    .select("pickup_location_key")
    .distinct()
)

In [0]:
display(
    df_fact
    .join(
        df_location,
        df_fact.dropoff_location_key == df_location.location_key,
        "left"
    )
    .filter(df_location.location_key.isNull())
    .select("dropoff_location_key")
    .distinct()
)

In [0]:
from pyspark.sql.functions import col

display(
    df_fact.alias("fact")
    .join(
        df_date.alias("date"),
        col("fact.date_key") == col("date.date_key"),
        "left"
    )
    .filter(col("date.date_key").isNull())
    .select(
        col("fact.date_key").alias("date_key")
    )
    .distinct()
)

In [0]:
display(
    df_fact
    .groupBy("provider_key")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
display(
    df_fact
    .groupBy("date_key")
    .count()
    .orderBy("date_key")
)

In [0]:
FACT_TABLE = "workspace.default.fact_trip"

(
    df_fact_trip
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("_source_month")
    .saveAsTable(FACT_TABLE)
)

In [0]:
from pyspark.sql.functions import col

SILVER_TABLE = "workspace.default.silver_hvfhv_trips"
DISPATCH_TABLE = "workspace.default.dim_dispatch"

df_silver = spark.table(SILVER_TABLE)
df_dispatch = spark.table(DISPATCH_TABLE)

In [0]:
dispatching_dim = (
    df_dispatch
    .select(
        col("base_num").alias("dispatching_base_num"),
        col("dispatch_key").alias("dispatching_base_key")
    )
)

originating_dim = (
    df_dispatch
    .select(
        col("base_num").alias("originating_base_num"),
        col("dispatch_key").alias("originating_base_key")
    )
)

In [0]:
df_fact_updated = (
    df_silver
    .join(
        dispatching_dim,
        on="dispatching_base_num",
        how="left"
    )
    .join(
        originating_dim,
        on="originating_base_num",
        how="left"
    )
)

In [0]:
from pyspark.sql.functions import (
    xxhash64,
    date_format,
    to_date
)

df_fact_updated = (
    df_fact_updated
    .withColumn(
        "trip_key",
        xxhash64(
            "hvfhs_license_num",
            "request_datetime",
            "pickup_datetime",
            "dropoff_datetime",
            "PULocationID",
            "DOLocationID",
            "trip_miles",
            "trip_time",
            "base_passenger_fare",
            "driver_pay"
        )
    )
    .withColumn(
        "date_key",
        date_format(
            to_date("pickup_datetime"),
            "yyyyMMdd"
        ).cast("integer")
    )
    .withColumn(
        "provider_key",
        col("hvfhs_license_num")
    )
    .withColumn(
        "pickup_location_key",
        col("PULocationID").cast("integer")
    )
    .withColumn(
        "dropoff_location_key",
        col("DOLocationID").cast("integer")
    )
)

In [0]:
fact_columns = [
    "trip_key",

    "date_key",
    "provider_key",
    "pickup_location_key",
    "dropoff_location_key",

    "dispatching_base_key",
    "originating_base_key",

    "request_datetime",
    "on_scene_datetime",
    "pickup_datetime",
    "dropoff_datetime",

    "trip_miles",
    "trip_time",

    "customer_wait_seconds",
    "driver_response_seconds",
    "calculated_trip_time_seconds",

    "base_passenger_fare",
    "tolls",
    "bcf",
    "sales_tax",
    "congestion_surcharge",
    "airport_fee",
    "tips",
    "driver_pay",
    "cbd_congestion_fee",

    "shared_request",
    "shared_match",
    "access_a_ride",
    "wav_request",
    "wav_match",

    "trip_time_consistent",
    "timestamp_quality_flag",
    "financial_quality_flag",
    "overall_quality_status",

    "_source_month",
    "_source_file",
    "_ingested_at"
]

df_fact_updated = df_fact_updated.select(fact_columns)

In [0]:
print("Rows:", df_fact_updated.count())
print("Columns:", len(df_fact_updated.columns))

In [0]:
display(
    df_fact_updated.select(
        "dispatching_base_key",
        "originating_base_key"
    ).limit(20)
)

In [0]:
print(
    "NULL originating base keys:",
    df_fact_updated
    .filter(col("originating_base_key").isNull())
    .count()
)

In [0]:
FACT_TABLE = "workspace.default.fact_trip"

(
    df_fact_updated
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_source_month")
    .saveAsTable(FACT_TABLE)
)

In [0]:
df_fact_final = spark.table(FACT_TABLE)

print("Persisted rows:", df_fact_final.count())
print("Persisted columns:", len(df_fact_final.columns))

In [0]:
display(
    df_fact_final.select(
        "trip_key",
        "date_key",
        "provider_key",
        "pickup_location_key",
        "dropoff_location_key",
        "dispatching_base_key",
        "originating_base_key"
    ).limit(10)
)